# Reproducible OD Pairs Processing

Build a route-specific locations table and hourly origin-destination flows from Transit Lab workbooks. Run the cells from top to bottom. Update the configuration below when changing the month, schedule, or carryover stop labels.

In [1]:
from pathlib import Path
import re
from datetime import time

import pandas as pd

PROJECT_ROOT = Path.cwd()
PATTERN_STOPS_PATH = PROJECT_ROOT / 'data' / 'pattern-stops' / '2025082420251018Sch-PatternStops_oct2025.csv'
TRANSIT_LAB_ROOT = PROJECT_ROOT / 'data' / 'transit-lab' / 'od-pairs'
MONTH_FOLDER = 'OCT2025'
DAY_TYPE = 'Weekdays'

ROUTES = {
    'BE': 'Buckeye Express',
    'CC': 'Campus Connector',
    'CLS': 'Campus Loop South',
    'ER': 'East Residential',
    'NWC': 'Northwest Connector',
    'MC': 'Med Center Express',
}

# Adjust for the route/workbook if its carryover labels differ.
CARRYOVER_STOP_ALIAS = {24: 1, 25: 2, 26: 3, 27: 4, 28: 5, 29: 6}

print(f'Project root: {PROJECT_ROOT}')
print(f'Pattern stops: {PATTERN_STOPS_PATH}')
print(f"Transit Lab data: {TRANSIT_LAB_ROOT / MONTH_FOLDER / 'Stop level' / 'Hourly' / DAY_TYPE}")

Project root: k:\AP\TTM\Data\transportation-planning
Pattern stops: k:\AP\TTM\Data\transportation-planning\data\pattern-stops\2025082420251018Sch-PatternStops_oct2025.csv
Transit Lab data: k:\AP\TTM\Data\transportation-planning\data\transit-lab\od-pairs\OCT2025\Stop level\Hourly\Weekdays


In [2]:
def clean_stop_id(value):
    if pd.isna(value):
        return None
    value_str = str(value).strip()
    if not value_str or 'total' in value_str.lower() or 'sum' in value_str.lower():
        return None
    if '(' in value_str:
        value_str = value_str.split('(', 1)[0].strip()
    if value_str.endswith('.0'):
        value_str = value_str[:-2]
    return value_str or None


def is_stop_label(value):
    cleaned = clean_stop_id(value)
    if cleaned is None or cleaned.lower() in {'origin', 'destination'}:
        return False
    try:
        float(cleaned)
        return True
    except ValueError:
        return False


def sort_stop_key(value):
    try:
        return int(float(clean_stop_id(value)))
    except (TypeError, ValueError):
        return 999999


def format_od_matrix(raw_df):
    origin_cells = [(r, c) for r in range(raw_df.shape[0]) for c in range(raw_df.shape[1])
                    if str(raw_df.iat[r, c]).strip().lower() == 'origin']
    destination_cells = [(r, c) for r in range(raw_df.shape[0]) for c in range(raw_df.shape[1])
                         if str(raw_df.iat[r, c]).strip().lower() == 'destination']
    if not origin_cells or not destination_cells:
        return pd.DataFrame()

    origin_row, origin_col = origin_cells[0]
    destination_header_row = destination_cells[0][0] + 1
    origin_id_col = next((col for col in range(origin_col, min(origin_col + 25, raw_df.shape[1]))
                          if is_stop_label(raw_df.iat[origin_row, col])), None)
    destination_start_col = next((col for col in range(origin_col + 1, min(origin_col + 40, raw_df.shape[1]))
                                  if is_stop_label(raw_df.iat[destination_header_row, col])), None)
    if origin_id_col is None or destination_start_col is None:
        return pd.DataFrame()

    destination_ids = {col: clean_stop_id(raw_df.iat[destination_header_row, col])
                       for col in range(destination_start_col, raw_df.shape[1])
                       if is_stop_label(raw_df.iat[destination_header_row, col])}
    rows = []
    for row_index in range(origin_row, raw_df.shape[0]):
        origin_value = raw_df.iat[row_index, origin_id_col]
        if pd.isna(origin_value):
            continue
        if 'total' in str(origin_value).lower() or 'sum' in str(origin_value).lower():
            break
        origin_id = clean_stop_id(origin_value)
        if origin_id is None or origin_id.lower() in {'origin', 'destination'}:
            continue

        matrix_row = {'Origin_ID': origin_id}
        for col, destination_id in destination_ids.items():
            try:
                volume = float(raw_df.iat[row_index, col])
            except (TypeError, ValueError):
                volume = 0.0
            matrix_row[destination_id] = volume if volume > 0 else 0.0
        rows.append(matrix_row)

    if not rows:
        return pd.DataFrame()
    matrix = pd.DataFrame(rows).set_index('Origin_ID').fillna(0)
    matrix = matrix.loc[:, ~matrix.columns.duplicated()]
    matrix = matrix.apply(pd.to_numeric, errors='coerce').fillna(0)
    matrix = matrix.sort_index(key=lambda index: [sort_stop_key(value) for value in index])
    matrix = matrix.reindex(sorted(matrix.columns, key=sort_stop_key), axis=1)
    matrix.index.name = 'Origin_ID'
    matrix.columns.name = 'Destination_ID'
    return matrix


def normalize_stop_num(value, aliases):
    cleaned = clean_stop_id(value)
    if cleaned is None:
        return value
    try:
        stop_num = int(float(cleaned))
    except ValueError:
        return value
    return aliases.get(stop_num, stop_num)


def matrix_to_long(matrix):
    if matrix.empty:
        return pd.DataFrame(columns=['origin_id', 'destination_id', 'volume'])
    long_df = matrix.copy()
    long_df.index = long_df.index.map(lambda value: str(value).strip())
    long_df.columns = long_df.columns.map(lambda value: str(value).strip())
    long_df = long_df.stack().reset_index()
    long_df.columns = ['origin_id', 'destination_id', 'volume']
    return long_df


def sheet_name_to_time(sheet_name):
    match = re.search(r'(\d{1,2})(AM|PM)-', sheet_name, flags=re.IGNORECASE)
    if not match:
        raise ValueError(f'Could not read an hour from worksheet name: {sheet_name}')
    hour, period = int(match.group(1)), match.group(2).upper()
    hour = (0 if hour == 12 else hour) if period == 'AM' else (hour if hour == 12 else hour + 12)
    return time(hour=hour)

In [ ]:
print('Route IDs:')
for route_code, route_name in ROUTES.items():
    print(f'  {route_code}: {route_name}')
route_id = input('Enter route ID: ').strip().upper()
if route_id not in ROUTES:
    raise ValueError(f'Invalid route ID: {route_id}. Choose one of {list(ROUTES)}.')
print(f'\n{ROUTES[route_id]} selected')

if not PATTERN_STOPS_PATH.is_file():
    raise FileNotFoundError(f'Pattern stops file not found: {PATTERN_STOPS_PATH}')
pattern_stops = pd.read_csv(PATTERN_STOPS_PATH, header=None)
kept_columns = [0, 1, 9, 10, 16, 17, 19]
if pattern_stops.shape[1] <= max(kept_columns):
    raise ValueError('Pattern stops file does not have the expected column layout.')
pattern_stops = pattern_stops.iloc[:, kept_columns].copy()
pattern_stops.columns = ['route_id', 'route_name', 'stop_id', 'stop_name', 'long', 'lat', 'direction']
selected_locations = pattern_stops.loc[
    pattern_stops['route_id'].astype(str).str.upper() == route_id
].drop_duplicates().drop(columns='direction').drop_duplicates().reset_index(drop=True)
if selected_locations.empty:
    raise ValueError(f'No pattern stops found for route {route_id}.')

display(selected_locations)
input('Review the locations, then press Enter to continue...')

hourly_data_dir = TRANSIT_LAB_ROOT / MONTH_FOLDER / 'Stop level' / 'Hourly' / DAY_TYPE
if not hourly_data_dir.is_dir():
    raise FileNotFoundError(f'OD data folder not found: {hourly_data_dir}')
route_token = re.compile(rf'(?<![A-Z0-9]){re.escape(route_id)}(?![A-Z0-9])', re.IGNORECASE)
candidate_files = sorted(
    path for path in hourly_data_dir.glob('*.xlsx')
    if route_token.search(path.stem) and not path.name.startswith('~$')
)
if len(candidate_files) != 1:
    raise FileNotFoundError(
        f'Expected one Excel file for route {route_id} in {hourly_data_dir}; found {len(candidate_files)}: '
        f'{[path.name for path in candidate_files]}'
    )
weekday_od_file = candidate_files[0]
print(f'Using OD workbook: {weekday_od_file.name}')

stop_summary = pd.read_excel(weekday_od_file, sheet_name='Stop Summary', header=4)
stop_summary = stop_summary.loc[:, ~stop_summary.columns.astype(str).str.startswith('Unnamed:')]
required_columns = {'Stop No.', 'Name'}
if not required_columns.issubset(stop_summary.columns):
    raise ValueError(f'Stop Summary is missing columns: {required_columns - set(stop_summary.columns)}')
carryover_rows = stop_summary['Stop No.'].astype('string').str.contains(r'\(\d+\)', regex=True, na=False)
if carryover_rows.any():
    first_carryover_position = carryover_rows.to_numpy().argmax()
    stop_summary = stop_summary.iloc[:first_carryover_position].copy()
stop_summary = stop_summary.rename(columns={'Stop No.': 'stop_num', 'Name': 'stop_name'})

print('Transit Lab stop summary:')
display(stop_summary)
removal_text = input('Stop number(s) to remove, comma-separated (press Enter for none): ').strip()
if removal_text:
    removal_ids = {clean_stop_id(item) for item in removal_text.split(',')}
    stop_summary = stop_summary.loc[
        ~stop_summary['stop_num'].map(clean_stop_id).isin(removal_ids)
    ].copy()

stop_summary = stop_summary.reset_index(drop=True)
shift = int(input('Number of positions to shift rows up (0 for no shift): ') or 0)
if len(stop_summary) and shift:
    shift %= len(stop_summary)
    stop_summary = pd.concat(
        [stop_summary.iloc[shift:], stop_summary.iloc[:shift]], ignore_index=True
    )

if len(selected_locations) != len(stop_summary):
    raise ValueError(
        f'Cannot pair stops by order: pattern stops has {len(selected_locations)} rows, '
        f'Transit Lab stop summary has {len(stop_summary)} rows.'
    )
ordered_stop_mapping = pd.concat(
    [
        selected_locations[['route_id', 'stop_id', 'stop_name', 'long', 'lat']].rename(
            columns={'stop_name': 'pattern_stop_name'}
        ),
        stop_summary[['stop_num', 'stop_name']].rename(
            columns={'stop_name': 'transit_lab_stop_name'}
        ),
    ],
    axis=1,
).reset_index(drop=True)
print('Review the ordered mapping before processing hourly flows:')
display(ordered_stop_mapping)
input('Press Enter to continue...')

# save ordered stop mapping to a CSV file
ordered_stop_mapping_output_path = input('Ordered stop mapping CSV path (press Enter to skip saving): ').strip()
if ordered_stop_mapping_output_path:
    ordered_stop_mapping_path = Path(ordered_stop_mapping_output_path)
    ordered_stop_mapping_path.parent.mkdir(parents=True, exist_ok=True)
    ordered_stop_mapping.to_csv(ordered_stop_mapping_path, index=False)
    print(f'Ordered stop mapping saved to {ordered_stop_mapping_path}')

Route IDs:
  BE: Buckeye Express
  CC: Campus Connector
  CLS: Campus Loop South
  ER: East Residential
  NWC: Northwest Connector
  MC: Med Center Express

Buckeye Express selected


,route_id,route_name,stop_id,stop_name,long,lat
0,BE,Buckeye Express,75,BUCKEYE LOT LOOP,-83.030646,40.015933
1,BE,Buckeye Express,20,MID WEST CAMPUS (EB),-83.026541,40.004251
2,BE,Buckeye Express,21,ST. JOHN EAST BOUND,-83.020744,40.004048
3,BE,Buckeye Express,57,KNOWLTON HALL,-83.016142,40.003979
4,BE,Buckeye Express,58,FONTANA LAB,-83.013197,40.003839
5,BE,Buckeye Express,605,HIGH AND 15TH,-83.008128,40.000207
6,BE,Buckeye Express,608,HIGH AND CHITTENDEN (SB),-83.007288,39.995934
7,BE,Buckeye Express,43,OHIO UNION (NB),-83.009564,39.997858
8,BE,Buckeye Express,62,ARPS HALL,-83.010531,40.002478
9,BE,Buckeye Express,55,BLACKBURN HOUSE,-83.012461,40.004004


Using OD workbook: OD flows inferred from APC data for BE October 2025_2025-10-01_2025-10-31.xlsx
Transit Lab stop summary:


,stop_num,stop_name
0,1,Fred Taylor & Schottenstein Drive
1,2,Buckeye Lot Loop
2,3,AG Campus (EB)
3,4,St John Arena (EB)
4,5,Knowlton Hall
5,6,Fontana Lab
6,7,High and 15th
7,8,High and Chittenden
8,9,Ohio Union (NB)
9,10,12th & College Rd (EB)


Review the ordered mapping before processing hourly flows:


,route_id,stop_id,pattern_stop_name,long,lat,stop_num,transit_lab_stop_name
0,BE,75,BUCKEYE LOT LOOP,-83.030646,40.015933,2,Buckeye Lot Loop
1,BE,20,MID WEST CAMPUS (EB),-83.026541,40.004251,3,AG Campus (EB)
2,BE,21,ST. JOHN EAST BOUND,-83.020744,40.004048,4,St John Arena (EB)
3,BE,57,KNOWLTON HALL,-83.016142,40.003979,5,Knowlton Hall
4,BE,58,FONTANA LAB,-83.013197,40.003839,6,Fontana Lab
5,BE,605,HIGH AND 15TH,-83.008128,40.000207,7,High and 15th
6,BE,608,HIGH AND CHITTENDEN (SB),-83.007288,39.995934,8,High and Chittenden
7,BE,43,OHIO UNION (NB),-83.009564,39.997858,9,Ohio Union (NB)
8,BE,62,ARPS HALL,-83.010531,40.002478,11,Arps Garage
9,BE,55,BLACKBURN HOUSE,-83.012461,40.004004,12,Blackburn


''

In [7]:
with pd.ExcelFile(weekday_od_file) as workbook:
    hourly_sheets = [
        sheet for sheet in workbook.sheet_names
        if re.search(r'\d{1,2}(?:AM|PM)-\d{1,2}(?:AM|PM)', sheet, re.IGNORECASE)
    ]

hourly_long_frames = []
for sheet_name in hourly_sheets:
    raw_matrix = pd.read_excel(weekday_od_file, sheet_name=sheet_name, header=None)
    od_matrix = format_od_matrix(raw_matrix)
    if od_matrix.empty:
        print(f'Skipping worksheet without a readable OD matrix: {sheet_name}')
        continue

    od_matrix.index = [normalize_stop_num(value, CARRYOVER_STOP_ALIAS) for value in od_matrix.index]
    od_matrix.columns = [normalize_stop_num(value, CARRYOVER_STOP_ALIAS) for value in od_matrix.columns]
    od_matrix = od_matrix.groupby(level=0).sum()
    od_matrix = od_matrix.T.groupby(level=0).sum().T

    sheet_long = matrix_to_long(od_matrix)
    sheet_long['origin_id'] = sheet_long['origin_id'].astype(str)
    sheet_long['destination_id'] = sheet_long['destination_id'].astype(str)
    sheet_long = sheet_long.groupby(
        ['origin_id', 'destination_id'], as_index=False
    )['volume'].sum()
    sheet_long['time'] = sheet_name_to_time(sheet_name)
    sheet_long['route_id'] = route_id
    hourly_long_frames.append(sheet_long)

if not hourly_long_frames:
    raise ValueError(f'No readable hourly OD matrices found in {weekday_od_file.name}.')
hourly_long_df = pd.concat(hourly_long_frames, ignore_index=True)
print(f'Created {len(hourly_long_df):,} hourly OD rows across {len(hourly_long_frames)} worksheets.')
display(hourly_long_df.head(25))

flows_output_path = input('Hourly flows CSV path (press Enter to skip saving): ').strip()
if flows_output_path:
    flows_path = Path(flows_output_path)
    flows_path.parent.mkdir(parents=True, exist_ok=True)
    hourly_long_df.to_csv(flows_path, index=False)
    print(f'Hourly flows saved to {flows_path}')

hourly_long_df

Created 4,080 hourly OD rows across 16 worksheets.


,origin_id,destination_id,volume,time,route_id
0,1,1,0.0,06:00:00,BE
1,1,10,0.0,06:00:00,BE
2,1,11,0.0,06:00:00,BE
3,1,12,0.0,06:00:00,BE
4,1,13,0.0,06:00:00,BE
5,1,14,0.0,06:00:00,BE
6,1,15,0.0,06:00:00,BE
7,1,16,0.0,06:00:00,BE
8,1,17,0.0,06:00:00,BE
9,1,2,0.0,06:00:00,BE


,origin_id,destination_id,volume,time,route_id
0,1,1,0.0,06:00:00,BE
1,1,10,0.0,06:00:00,BE
2,1,11,0.0,06:00:00,BE
3,1,12,0.0,06:00:00,BE
4,1,13,0.0,06:00:00,BE
...,...,...,...,...,...
4075,9,5,0.0,21:00:00,BE
4076,9,6,0.0,21:00:00,BE
4077,9,7,0.0,21:00:00,BE
4078,9,8,0.0,21:00:00,BE
